# ParaEMT forced-oscillation analysis

This notebook analyzes this run's response to the forced oscillation: center-of-inertia (COI) frequencies
for all 14 zones, representative generators in each zone, the response of the forced source generator, the
regional response at the forcing frequency, and (for 20 Hz cases) IBR PLL frequencies at buses chosen by
topological distance from the source. It also writes derived tables and validation results.

The RAW and XLSX model files are read from the ParaEMT copy in this run's `build/` directory. Generated files
are written to `notebook_outputs/` in the run directory.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import posixpath
import re
import sys
import xml.etree.ElementTree as ET
import zipfile
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.feather as pf
from IPython.display import Markdown, display

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / 'notebook_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ParaEMT's default start time when simulation.oedi_start_timestamp is unset.
DEFAULT_START_TIMESTAMP = '2026-01-01 00:00:00'

# Figures produced by the analysis, keyed by output name, for display below.
FIGURES = {}

print(f'Run:        {RUN_DIR}')
print(f'Output:     {OUTPUT_DIR}')
print(f'Python:     {sys.executable}')

## Recorded streams

The analysis uses five ParaEMT streams. The run's `wiring.json` links say which Recorder holds each one, so
Recorder names do not matter. Timestamps are converted to seconds from the start of the simulation.

In [ ]:
required_streams = {
    3: ('ibr_frequency', 'IBR/PLL frequency'),
    4: ('bus_electrical_power', 'bus electrical power'),
    5: ('generator_electrical_power', 'generator electrical power'),
    6: ('generator_reactive_power', 'generator reactive power'),
    8: ('generator_rotor_speed', 'generator rotor speed'),
}

wiring = json.loads((RUN_DIR / 'wiring.json').read_text(encoding='utf-8'))
components = {component['name']: component for component in wiring['components']}
paraemt = [
    component for component in wiring['components']
    if 'simulation' in component.get('parameters', {})
    and 'powerflow' in component.get('parameters', {})
]
if len(paraemt) != 1:
    raise ValueError(f'Expected one ParaEMT component in wiring.json; found {len(paraemt)}.')
paraemt = paraemt[0]
PARAEMT_BUILD_DIR = RUN_DIR / 'build' / paraemt['name']
MODEL_DIR = PARAEMT_BUILD_DIR / 'models' / '240bus_psse'

recorders = {}
for link in wiring['links']:
    target = components[link['target']]
    if link['source'] == paraemt['name'] and target['type'] == 'Recorder':
        recorders[link['source_port']] = target

start = pd.Timestamp(paraemt['parameters']['simulation'].get('oedi_start_timestamp', DEFAULT_START_TIMESTAMP))
STREAM_FILES, stream_frames, stream_rows = {}, {}, []
for number, (stream, description) in required_streams.items():
    if stream not in recorders:
        raise KeyError(f'No Recorder is attached to ParaEMT stream {stream!r}. Add one in the frontend and run again.')
    recorder = recorders[stream]
    path = RUN_DIR / 'build' / recorder['name'] / recorder['parameters']['feather_filename']
    if not path.is_file():
        raise FileNotFoundError(f'{stream}: {path} does not exist.')
    frame = pf.read_feather(path)
    frame['time'] = (pd.to_datetime(frame['time']) - start).dt.total_seconds()
    frame = frame[['time', *frame.columns.drop('time')]]
    STREAM_FILES[number], stream_frames[number] = path, frame
    stream_rows.append({
        'stream': stream,
        'description': description,
        'file': str(path.relative_to(RUN_DIR)),
        'rows': len(frame),
        'time_s': f"{frame['time'].iloc[0]:.3f}-{frame['time'].iloc[-1]:.3f}",
        'size_MB': path.stat().st_size / 1_000_000,
    })

display(Markdown('## Scenario configuration'))
display(pd.Series(paraemt['parameters']['forced_oscillation'], name='value').to_frame())
display(Markdown('## Required recorded streams'))
display(pd.DataFrame(stream_rows).set_index('stream'))

## Analysis code

The scenario configuration is read from the `static_inputs.json` that ParaEMT ran with, in this run's build
directory.

In [ ]:
from pathlib import Path
import csv
import zipfile
import xml.etree.ElementTree as ET
import posixpath
import re
import numpy as np
import pandas as pd
# Set by configure() in the data-loading cell above.
DATA = ROOT = AREA_RAW = MACHINE_RAW = DYNAMIC_XLSX = None
STREAM_FRAMES = {}


def configure(data_dir=None, support_dir=None, stream_frames=None):
    """Configure scenario/support paths and optional in-memory HELICS frames."""
    global DATA, ROOT, AREA_RAW, MACHINE_RAW, DYNAMIC_XLSX, STREAM_FRAMES
    if data_dir is not None:
        DATA = Path(data_dir).resolve()
    ROOT = Path(support_dir).resolve() if support_dir is not None else DATA.parent
    AREA_RAW = ROOT / "WECC240_2600_2619IBR.raw"
    MACHINE_RAW = ROOT / "240busWECC_2018_PSS_PQLoad_fixedshunt_rm0load_2600_2619G_mod.raw"
    DYNAMIC_XLSX = MACHINE_RAW.with_suffix(".xlsx")
    STREAM_FRAMES = {
        int(number): frame.copy()
        for number, frame in (stream_frames or {}).items()
    }


def static_inputs_path():
    paths = sorted(DATA.glob("static_inputs*.json"))
    if len(paths) != 1:
        raise ValueError(f"Expected one static_inputs*.json in {DATA}, found {paths}")
    return paths[0]
def read_dynamic_generators(path):
    """Read the column-oriented gen sheet with stdlib OOXML, retaining cell provenance."""
    ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(path) as archive:
        strings = []
        if 'xl/sharedStrings.xml' in archive.namelist():
            strings = [''.join(node.itertext()) for node in
                       ET.fromstring(archive.read('xl/sharedStrings.xml')).findall('m:si', ns)]
        workbook = ET.fromstring(archive.read('xl/workbook.xml'))
        sheets = workbook.findall('m:sheets/m:sheet', ns)
        sheet = next(s for s in sheets if s.attrib['name'] == 'gen')
        rid = sheet.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']
        relationships = ET.fromstring(archive.read('xl/_rels/workbook.xml.rels'))
        target = next(r.attrib['Target'] for r in relationships if r.attrib['Id'] == rid)
        target = target.lstrip('/') if target.startswith('/') else posixpath.normpath('xl/' + target)
        cells, labels = {}, {}
        for row in ET.fromstring(archive.read(target)).findall('m:sheetData/m:row', ns):
            for cell in row.findall('m:c', ns):
                v = cell.find('m:v', ns)
                value = v.text if v is not None else ''
                if cell.attrib.get('t') == 's':
                    value = strings[int(value)]
                elif cell.attrib.get('t') == 'inlineStr':
                    value = ''.join(cell.find('m:is', ns).itertext())
                cells[cell.attrib['r']] = value
            label = cells.get('A' + row.attrib['r'], '').strip()
            labels[label] = row.attrib['r']
        records = {}
        for address, value in cells.items():
            match = re.fullmatch(r'([A-Z]+)' + labels['Bus #'], address)
            if not match or match[1] == 'A' or not value:
                continue
            col = match[1]
            key = (int(float(value)), cells[col + labels['Gen id']].strip())
            h_cell = col + labels['H']
            h = float(cells[h_cell])
            model = cells[col + labels['type']].strip()
            if key in records or not np.isfinite(h) or h <= 0 or model != 'GENROU':
                raise ValueError(f'Invalid or duplicate dynamic generator {key}')
            records[key] = dict(H_s=h, dynamic_model=model,
                                inertia_source=path.name, inertia_cell='gen!' + h_cell)
    return records

def read_raw(path):
    """Read bus, generator and area records from this PSS/E RAW file."""
    buses, machines, areas = {}, {}, {}
    section = 'bus'
    for line in path.read_text().splitlines()[3:]:
        if line.strip().startswith('0 /'):
            section = line.split('Begin ', 1)[-1].lower() if 'Begin ' in line else ''
            continue
        if not line.strip() or line.strip() == 'Q':
            continue
        if section != 'bus' and not section.startswith(('generator data', 'area interchange data')):
            continue
        v = next(csv.reader([line], quotechar="'", skipinitialspace=True))
        if section == 'bus':
            buses[int(v[0])] = dict(bus=int(v[0]), bus_name=v[1].strip(),
                                   kv=float(v[2]), area=int(v[4]), zone=int(v[5]))
        elif section.startswith('generator data'):
            machines[int(v[0]), v[1].strip()] = dict(raw_pg_mw=float(v[2]),
                                                    mbase_mva=float(v[8]), status=int(v[14]))
        else:
            areas[int(v[0])] = v[4].strip()
    return buses, machines, areas

def load_csv(number):
    if number in STREAM_FRAMES:
        df = STREAM_FRAMES[number].copy()
        if not np.isfinite(df.to_numpy(dtype=float)).all() or not (np.diff(df.time) > 0).all():
            raise ValueError(f'Nonfinite data or nonmonotonic time in HELICS stream {number}')
        return df
    raise KeyError(f'Stream {number} was not loaded from this run.')

In [ ]:
# Source-machine response, cycle means, spectra, and regional forcing response.
import json
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def harmonic_fit(t, y, hz):
    """Fit offset + linear trend + sin/cos; amplitude is peak, not peak-to-peak."""
    x = t-t[0]
    design = np.column_stack([np.ones(len(t)), x-x.mean(), np.sin(2*np.pi*hz*x), np.cos(2*np.pi*hz*x)])
    beta, _, rank, _ = np.linalg.lstsq(design, y, rcond=None)
    if rank != 4:
        raise ValueError('Singular harmonic fit')
    return dict(mean=float(beta[0]), slope_per_s=float(beta[1]),
                amplitude_peak=float(np.hypot(beta[2], beta[3])),
                residual_rms=float(np.sqrt(np.mean((y-design@beta)**2))))


def amplitude_spectrum(t, y):
    """Linearly detrended, Hann-windowed, coherent-gain corrected rFFT peak amplitude."""
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t), dt, atol=1e-9, rtol=0):
        raise ValueError('FFT requires uniformly sampled time')
    x=t-t.mean()
    design=np.column_stack([np.ones(len(t)), x])
    residual=y-design@np.linalg.lstsq(design,y,rcond=None)[0]
    window=np.hanning(len(t))
    amplitude=2*np.abs(np.fft.rfft(residual*window))/window.sum()
    amplitude[0]*=.5
    if len(t)%2==0:
        amplitude[-1]*=.5
    return np.fft.rfftfreq(len(t),dt), amplitude


def governor_metadata():
    """Export original generic governor row labels without reinterpreting them."""
    ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(DYNAMIC_XLSX) as archive:
        strings=[''.join(x.itertext()) for x in ET.fromstring(archive.read('xl/sharedStrings.xml')).findall('m:si',ns)]
        workbook=ET.fromstring(archive.read('xl/workbook.xml'))
        sheet=next(s for s in workbook.findall('m:sheets/m:sheet',ns) if s.attrib['name']=='gov')
        rid=sheet.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']
        rels=ET.fromstring(archive.read('xl/_rels/workbook.xml.rels'))
        target=next(r.attrib['Target'] for r in rels if r.attrib['Id']==rid)
        path=target.lstrip('/') if target.startswith('/') else 'xl/'+target
        cells={}
        for cell in ET.fromstring(archive.read(path)).findall('m:sheetData/m:row/m:c',ns):
            v=cell.find('m:v',ns)
            value=v.text if v is not None else ''
            cells[cell.attrib['r']]=strings[int(value)] if cell.attrib.get('t')=='s' else value
        col=next(address[:-1] for address,value in cells.items()
                 if address.endswith('1') and address[:-1].isalpha() and value=='4131'
                 and cells.get(address[:-1]+'2')=='H')
        records=[dict(sheet='gov', source_cell=col+str(i),
                      original_row_label=cells['A'+str(i)], value=cells.get(col+str(i),''))
                 for i in range(1,100) if 'A'+str(i) in cells]
        if cells[col+'3']!='HYGOV':
            raise ValueError('Source governor is not HYGOV')
        return records


def create_diagnostics(t,power,speed,frequency,coi,meta,target,fo,config,out):
    start,hz=float(fo['start_time']),float(fo['frequency_hz'])
    hz_label=f'{hz:g}'
    coi_amplitude_column=f'coi_{hz_label}Hz_peak_mHz'
    representative_amplitude_column=f'representative_{hz_label}Hz_peak_mHz'
    pre=(t>=start-.2)&(t<start)
    fitmask=(t>=2.)&(t<5.)
    tt=t[fitmask]
    if not np.any(pre) or len(tt)<100:
        raise ValueError('Insufficient source-response record')
    q=load_csv(6)
    busp=load_csv(4)
    if not np.array_equal(q.time,t) or not np.array_equal(busp.time,t):
        raise ValueError('Source-response CSV time mismatch')
    source_channels=[c for c in power if '_bus4131_' in c]
    source_zone=int(meta.loc[target,'zone'])
    assert source_zone==13 and meta.loc[target,'zone_name']=='WASHINGTON'
    zonecol=next(c for c in coi if c.startswith(f'zone{source_zone:02d}_'))
    p0=float(power.loc[pre,target].mean())
    q0=float(q.loc[pre,target].mean())
    f0=float(frequency.loc[pre,target].mean())
    ids=np.floor((t-start)*hz+1e-8).astype(int)
    expected_samples=int(round(1/(hz*np.median(np.diff(t)))))
    cycles=[]
    cycle_mean_series=np.full(len(t),np.nan)
    for k in range(int(np.floor((t[-1]+np.median(np.diff(t))-start)*hz+1e-8))):
        mask=ids==k
        if mask.sum()!=expected_samples:
            raise ValueError(f'Incomplete forcing cycle {k}: {mask.sum()} samples')
        mean=float(power.loc[mask,target].mean())
        cycle_mean_series[mask]=mean
        cycles.append(dict(cycle=k+1,start_s=start+k/hz,end_s=start+(k+1)/hz,
                           midpoint_s=start+(k+.5)/hz,samples=int(mask.sum()),
                           target_P_mean_MW=mean,delta_P_mean_MW=mean-p0,
                           target_f_mean_Hz=float(frequency.loc[mask,target].mean()),
                           target_Q_mean_Mvar=float(q.loc[mask,target].mean())))
    cycle_table=pd.DataFrame(cycles)
    continuous_mean=np.interp(t,cycle_table.midpoint_s,cycle_table.target_P_mean_MW)
    continuous_mean[(t<cycle_table.midpoint_s.iloc[0])|(t>cycle_table.midpoint_s.iloc[-1])]=np.nan
    cycle_table.to_csv(out/'source_cycle_means.csv',index=False)
    rows=governor_metadata()
    pd.DataFrame(rows).to_csv(out/'source_governor_parameters.csv',index=False)
    source=pd.DataFrame({'time_s':t,'target_P_MW':power[target],'target_Q_Mvar':q[target],
                         'target_rotor_frequency_Hz':frequency[target],
                         'zone13_coi_Hz':coi[zonecol], 'bus4131_total_P_MW':busp['4131'],
                         'target_delta_P_MW':power[target]-p0,
                         'target_cycle_mean_P_MW':cycle_mean_series,
                         'target_interpolated_cycle_mean_P_MW':continuous_mean,
                         'target_P_minus_interpolated_cycle_mean_MW':power[target]-continuous_mean,
                         'target_P_minus_cycle_mean_MW':power[target]-cycle_mean_series})
    for c in source_channels:
        source[c+'_P_MW']=power[c]
        source[c+'_rotor_Hz']=frequency[c]
    source.to_csv(out/'bus4131_source_response.csv',index=False)
    if not np.allclose(power[source_channels].sum(axis=1),busp['4131'],rtol=0,atol=1e-6):
        raise ValueError('Bus 4131 power does not match sum of recorded generators')
    pfit=harmonic_fit(tt,power.loc[fitmask,target].to_numpy(),hz)
    ffit=harmonic_fit(tt,frequency.loc[fitmask,target].to_numpy(),hz)
    fft_hz,fft_p=amplitude_spectrum(tt,power.loc[fitmask,target].to_numpy())
    _,fft_f=amplitude_spectrum(tt,frequency.loc[fitmask,target].to_numpy())
    pd.DataFrame({'frequency_Hz':fft_hz,'target_P_peak_MW':fft_p,
                  'target_rotor_frequency_peak_Hz':fft_f}).to_csv(out/'source_amplitude_spectra.csv',index=False)
    regional=[]
    for col in coi.columns.drop('time_s'):
        z=int(col[4:6]); fit=harmonic_fit(tt,coi.loc[fitmask,col].to_numpy(),hz)
        rep=meta[(meta.zone==z)&(meta.kv==20)].sort_values(['pre_event_p_mw','column'],ascending=[False,True]).iloc[0]
        rf=harmonic_fit(tt,frequency.loc[fitmask,rep.column].to_numpy(),hz)
        row=dict(zone=z,zone_name=rep.zone_name,representative=rep.column)
        row[coi_amplitude_column]=1000*fit['amplitude_peak']
        row[representative_amplitude_column]=1000*rf['amplitude_peak']
        regional.append(row)
    pd.DataFrame(regional).to_csv(out/f'regional_{hz_label}Hz_response.csv',index=False)
    def save(fig,name):
        fig.savefig(out/(name+'.png'),dpi=200,bbox_inches='tight')
        fig.savefig(out/(name+'.pdf'),bbox_inches='tight')
        FIGURES[name]=fig
        plt.close(fig)
    def axes_format(ax,ylabel):
        ax.axvline(start,color='#BA3945',ls='--',lw=1,label=f'Forcing starts at {start:g} s')
        ax.set(xlabel='Time (s)',ylabel=ylabel,xlim=(0,t[-1]))
    fig,ax=plt.subplots(figsize=(11,5),layout='constrained')
    ax.plot(t,power[target],color='#147D92',ls='-',
            label='Generator 66 / Bus 4131 / ID H')
    ax.axvline(start,color='#BA3945',ls='-',lw=1.2,
               label=f'Forcing starts at {start:g} s')
    ax.set(xlabel='Time (s)',ylabel='Active power (MW)',xlim=(0,t[-1]))
    ax.ticklabel_format(axis='y',style='plain',useOffset=False)
    ax.set_title('COULEE generator active power | Zone 13 WASHINGTON',
                 loc='left',fontweight='bold')
    ax.legend(loc='best')
    save(fig,'08_gen66_active_power_vs_time')
    pd.DataFrame({'time_s':t,'generator':target,'active_power_MW':power[target]}).to_csv(
        out/'gen66_active_power_vs_time.csv',index=False)
    fig,axes=plt.subplots(3,1,figsize=(12,10),layout='constrained')
    for c in source_channels:
        baseline=float(power.loc[pre,c].mean())
        axes[0].plot(t,power[c]-baseline,label=c)
    axes[0].plot(cycle_table.midpoint_s,cycle_table.delta_P_mean_MW,color='#20252B',ls='--',label='Gen66 cycle mean')
    axes_format(axes[0],'Active power change (MW)')
    axes[1].plot(t,q[target]-q0,color='#D88120',label='Gen66 reactive power change')
    axes_format(axes[1],'Reactive power change (Mvar)')
    for c in source_channels:
        axes[2].plot(t,(frequency[c]-float(frequency.loc[pre,c].mean()))*1000,label=c)
    axes[2].plot(t,(coi[zonecol]-float(coi.loc[pre,zonecol].mean()))*1000,color='#20252B',ls='--',label='Zone 13 COI')
    axes_format(axes[2],'Frequency change (mHz)')
    for ax in axes:ax.legend(loc='best',fontsize=8)
    fig.suptitle('COULEE: bus 4131, generator 66 / ID H | RAW Zone 13 WASHINGTON',fontweight='bold')
    save(fig,'04_bus4131_source_response')
    fig,axes=plt.subplots(3,1,figsize=(12,10),layout='constrained')
    axes[0].plot(t,power[target]-p0,color='#147D92',label='Gen66 active power change')
    axes[0].plot(cycle_table.midpoint_s,cycle_table.delta_P_mean_MW,'o--',color='#BA3945',ms=3,
                 label=f'Complete {1/hz:g} s cycle mean')
    axes_format(axes[0],'Delta P (MW)')
    axes[1].plot(cycle_table.midpoint_s,cycle_table.delta_P_mean_MW,'o-',color='#BA3945')
    axes[1].axhline(0,color='#555555',lw=1)
    axes[1].set(xlabel='Cycle midpoint (s)',ylabel='Cycle-mean delta P (MW)',xlim=(start,5))
    zoom=(t>=4)&(t<=4.9)
    axes[2].plot(t[zoom],(power[target]-continuous_mean)[zoom],color='#147D92',label='P minus interpolated cycle means')
    axes[2].axhline(0,color='#555555',lw=1)
    axes[2].set(xlabel='Time (s)',ylabel='Residual active power (MW)',xlim=(4,4.9))
    for ax in [axes[0],axes[2]]:ax.legend(loc='best',fontsize=9)
    fig.suptitle('Gen66: oscillatory response and observed mean-power shift',fontweight='bold')
    save(fig,'05_gen66_cycle_mean_and_oscillation')
    fig,axes=plt.subplots(2,1,figsize=(12,8),layout='constrained')
    for ax,vals,label,unit,fit in [(axes[0],fft_p,'Electrical active power','MW',pfit),
                                  (axes[1],fft_f*1000,'Rotor frequency','mHz',ffit)]:
        ax.plot(fft_hz,vals,color='#147D92',label='Detrended, Hann-windowed spectrum')
        ax.axvline(hz,color='#BA3945',ls='--',label=f'Configured forcing: {hz_label} Hz')
        ax.set(xlabel='Frequency (Hz)',ylabel=f'Peak amplitude ({unit})',xlim=(0,max(20,2*hz)))
        amp=fit['amplitude_peak']*(1000 if unit=='mHz' else 1)
        ax.set_title(f'{label} | {hz_label} Hz sine-fit amplitude = {amp:.6g} {unit}',loc='left',fontsize=11)
        ax.legend(loc='upper right',fontsize=9)
    fig.suptitle('Gen66 response spectra | 2.000 <= t < 5.000 s | resolution 0.333 Hz',fontweight='bold')
    save(fig,'06_gen66_response_spectra')
    fig,ax=plt.subplots(figsize=(12,7),layout='constrained')
    regional_df=pd.DataFrame(regional);loc=np.arange(14)
    ax.barh(loc-.18,regional_df[coi_amplitude_column],height=.35,label='Zone COI',color='#147D92')
    ax.barh(loc+.18,regional_df[representative_amplitude_column],height=.35,label='20 kV representative',color='#D88120')
    ax.set_yticks(loc,[f'{r["zone"]:02d} {r["zone_name"]}' for r in regional])
    ax.invert_yaxis();ax.set(xlabel=f'{hz_label} Hz component peak amplitude (mHz)',
                             title=f'Regional {hz_label} Hz response | sine/cosine + linear trend fit, 2-5 s')
    ax.legend(loc='best')
    save(fig,f'07_regional_{hz_label}Hz_response')
    late=cycle_table[cycle_table.start_s>=4]
    band=(fft_hz>=1)&(fft_hz<=20)
    report=dict(target=target,bus=4131,zone=source_zone,zone_name='WASHINGTON',governor_model='HYGOV',
                forcing_parameters=fo,baseline_window_s=[start-.2,start],baseline_interval='left closed, right open',
                pre_event_P_MW=p0,late_cycle_mean_P_MW=float(late.target_P_mean_MW.mean()),
                observed_late_mean_shift_MW=float(late.delta_P_mean_MW.mean()),
                late_window_s=[4,5],complete_forcing_cycles=len(cycles),samples_per_cycle=expected_samples,
                harmonic_fit_window_s=[2,5],harmonic_fit_P=pfit,harmonic_fit_rotor_frequency=ffit,
                spectral_resolution_Hz=float(fft_hz[1]-fft_hz[0]),
                strongest_P_spectral_bin_in_1_to_20_Hz=float(fft_hz[band][np.argmax(fft_p[band])]),
                mechanism='User-provided explanation: HYGOV rate limiter and nonlinear turbine dynamics can rectify the zero-mean forcing.',
                evidence_limit='Recorded electrical P,Q and rotor speed demonstrate response and mean shift; no gate, mechanical power, water-flow, limiter-state or injected-signal traces are available to independently establish that causal mechanism.',
                forcing_magnitude_units='Not defined by JSON; magnitude 0.1 is not converted to MW or labeled pu.',
                raw_zone_authoritative=True,bus_power_equals_sum_recorded_generators=True)
    (out/'source_summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(json.dumps(report,indent=2))

In [ ]:
# BFS topology distances and representative IBR-bus PLL plots for the 20 Hz case.
from pathlib import Path
from collections import deque
import csv
import hashlib
import json
import posixpath
import zipfile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def read_topology(raw_path):
    buses, zones, edges = {}, {}, []
    section = 'bus'
    lines = raw_path.read_text().splitlines()[3:]
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        i += 1
        if line.startswith('0 /'):
            section = line.split('Begin ', 1)[-1].lower() if 'Begin ' in line else ''
            continue
        if not line or line == 'Q':
            continue
        v = next(csv.reader([line], quotechar="'", skipinitialspace=True))
        if section == 'bus':
            buses[int(v[0])] = dict(bus=int(v[0]), bus_name=v[1].strip(), kv=float(v[2]), zone=int(v[5]))
        elif section.startswith('zone data'):
            zones[int(v[0])] = v[1].strip()
        elif section.startswith('branch data'):
            if int(v[13]) == 1:
                edges.append(dict(from_bus=abs(int(v[0])), to_bus=abs(int(v[1])), circuit=v[2].strip(), kind='line'))
        elif section.startswith('transformer data'):
            a, b, c = map(int, v[:3])
            status = int(v[11])
            if c != 0:
                raise ValueError('Three-winding transformer requires an explicit distance convention')
            if status == 1:
                edges.append(dict(from_bus=a, to_bus=b, circuit=v[3].strip(), kind='transformer'))
            i += 3  # two-winding transformer has four record lines in total
    return buses, zones, edges


def read_ibrs(xlsx_path):
    ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(xlsx_path) as z:
        strings = [''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si', ns)]
        wb = ET.fromstring(z.read('xl/workbook.xml'))
        rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
        records = []
        for sheet in wb.findall('m:sheets/m:sheet', ns):
            if sheet.attrib['name'] not in {'regca', 'ibrepri'}:
                continue
            rid = sheet.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']
            target = next(r.attrib['Target'] for r in rels if r.attrib['Id'] == rid)
            path = target.lstrip('/') if target.startswith('/') else posixpath.normpath('xl/' + target)
            cells = {}
            for cell in ET.fromstring(z.read(path)).findall('m:sheetData/m:row/m:c', ns):
                node = cell.find('m:v', ns)
                val = node.text if node is not None else ''
                if cell.attrib.get('t') == 's':
                    val = strings[int(val)]
                cells[cell.attrib['r']] = val
            for addr, value in cells.items():
                if addr.endswith('1') and addr[:-1].isalpha() and addr[:-1] != 'A' and value:
                    records.append(dict(bus=int(float(value)), generator_id=cells.get(addr[:-1]+'2', ''), model=sheet.attrib['name']))
    return records


def plot_representative_ibr_pll(output_dir):
    out = Path(output_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    raw_path = MACHINE_RAW
    xlsx_path = DYNAMIC_XLSX
    buses, zones, edges = read_topology(raw_path)
    graph = {b: set() for b in buses}
    for e in edges:
        a, b = e['from_bus'], e['to_bus']
        assert a in buses and b in buses
        graph[a].add(b)
        graph[b].add(a)
    distance, parent = {4131: 0}, {4131: None}
    queue = deque([4131])
    while queue:
        a = queue.popleft()
        for b in sorted(graph[a]):
            if b not in distance:
                distance[b], parent[b] = distance[a]+1, a
                queue.append(b)
    data = load_csv(3)
    assert np.isfinite(data.to_numpy()).all() and (np.diff(data.time) > 0).all()
    ibrs = read_ibrs(xlsx_path)
    _, machines, _ = read_raw(raw_path)
    assert all(machines[r['bus'], r['generator_id']]['status'] == 1 for r in ibrs)
    ids = {}
    for r in ibrs:
        ids.setdefault(r['bus'], []).append(r['generator_id'])
    def path(bus):
        result = [bus]
        while parent[result[-1]] is not None:
            result.append(parent[result[-1]])
        return result[::-1]
    rows = []
    for c in data.columns.drop('time'):
        b = int(c)
        rows.append(dict(**buses[b], zone_name=zones[buses[b]['zone']],
                         distance_branches=distance.get(b), has_ibr=b in ids,
                         ibr_ids=';'.join(ids.get(b, [])),
                         shortest_path=' -> '.join(map(str, path(b))) if b in distance else 'UNREACHABLE'))
    table = pd.DataFrame(rows).sort_values(['distance_branches', 'bus'])
    table.to_csv(out/'all_pll_bus_distances.csv', index=False)
    pd.DataFrame(edges).to_csv(out/'in_service_topology_edges.csv', index=False)
    candidates = table[table.has_ibr & table.distance_branches.notna()].copy()
    candidates.to_csv(out/'ibr_bus_candidates.csv', index=False)
    lo, hi = int(candidates.distance_branches.min()), int(candidates.distance_branches.max())
    chosen = []
    for role, subset in [('Near', candidates[candidates.distance_branches == lo]),
                         ('Middle', candidates[(candidates.distance_branches > lo) & (candidates.distance_branches < hi)]),
                         ('Far', candidates[candidates.distance_branches == hi])]:
        subset = subset.copy()
        subset['midpoint_error'] = abs(subset.distance_branches - (lo+hi)/2)
        row = subset.sort_values(['midpoint_error', 'bus']).iloc[0].drop(labels='midpoint_error').to_dict()
        row['role'] = role
        chosen.append(row)
    pd.DataFrame(chosen).to_csv(out/'selected_representatives.csv', index=False)
    selected = pd.DataFrame({'time_s': data.time})
    colors = ['#147D92', '#D88120', '#7455A4']
    plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.grid': True, 'grid.alpha': .22, 'pdf.fonttype': 42})
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), layout='constrained')
    pre = (data.time >= .8) & (data.time < 1)
    fitmask = (data.time >= 2) & (data.time < 5)
    for row, color in zip(chosen, colors):
        b = int(row['bus'])
        f = data[str(b)]
        baseline = float(f[pre].mean())
        delta = (f-baseline)*1000
        label = f"{row['role']}: Bus {b} | {int(row['distance_branches'])} branches | Zone {row['zone']} {row['zone_name']}"
        axes[0].plot(data.time, f, color=color, label=label)
        axes[1].plot(data.time, delta, color=color, label=f'Bus {b}')
        zoom = (data.time >= 4) & (data.time <= 4.25)
        axes[2].plot(data.time[zoom], delta[zoom], color=color, label=f'Bus {b}')
        selected[f'bus{b}_PLL_Hz'] = f
        row['pre_event_mean_hz'] = baseline
        row['fit_20Hz_peak_mHz'] = harmonic_fit(data.time[fitmask].to_numpy(), f[fitmask].to_numpy(), 20)['amplitude_peak']*1000
    axes[0].set(ylabel='PLL frequency (Hz)', xlim=(0,5), title='Absolute PLL frequency at representative IBR buses')
    axes[0].ticklabel_format(axis='y', style='plain', useOffset=False)
    axes[1].set(ylabel='Change from pre-event mean (mHz)', xlim=(0,5), title='Baseline: 0.8 <= t < 1.0 s')
    axes[2].set(ylabel='Change from pre-event mean (mHz)', xlim=(4,4.25), title='Five forcing periods: 4.00-4.25 s (includes slow response)')
    for ax in axes:
        ax.set_xlabel('Time (s)')
        ax.legend(loc='best', fontsize=8)
    for ax in axes[:2]:
        ax.axvline(1, color='#BA3945', ls='--', lw=1)
    fig.suptitle('20 Hz forced oscillation | Representative IBR bus PLL outputs\nDistance: BFS from Bus 4131, online lines + transformers', fontweight='bold')
    for suffix in ['png', 'pdf']:
        fig.savefig(out/f'01_near_middle_far_ibr_pll.{suffix}', dpi=200, bbox_inches='tight')
    FIGURES['representative_ibr_pll/01_near_middle_far_ibr_pll']=fig
    plt.close(fig)
    selected.to_csv(out/'selected_pll_frequency.csv', index=False)
    for b, d in distance.items():
        assert len(path(b))-1 == d
        assert all(v in graph[u] for u,v in zip(path(b), path(b)[1:]))
        assert all(abs(distance[n]-d) <= 1 for n in graph[b])
    reachable = table[table.distance_branches.notna()]
    extremes = {role: reachable[reachable.distance_branches == val].to_dict('records') for role,val in [('nearest', reachable.distance_branches.min()), ('farthest', reachable.distance_branches.max())]}
    # The furthest recorded PLL bus is not necessarily an IBR bus.
    all_mid = reachable[(reachable.distance_branches > 0) & (reachable.distance_branches < reachable.distance_branches.max())].copy()
    all_mid['midpoint_error'] = abs(all_mid.distance_branches-reachable.distance_branches.max()/2)
    middle_all = all_mid.sort_values(['midpoint_error', 'bus']).iloc[0].drop(labels='midpoint_error').to_dict()
    all_selected = [dict(extremes['nearest'][0], role='Near'), dict(middle_all, role='Middle'), dict(extremes['farthest'][0], role='Far')]
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), layout='constrained')
    all_export = pd.DataFrame({'time_s': data.time})
    for row, color in zip(all_selected, colors):
        b = int(row['bus'])
        f = data[str(b)]
        label = f"{row['role']}: Bus {b} | {int(row['distance_branches'])} branches | Zone {row['zone']} {row['zone_name']} | {'IBR' if row['has_ibr'] else 'no modeled IBR'}"
        axes[0].plot(data.time, f, color=color, label=label)
        axes[1].plot(data.time, (f-f[pre].mean())*1000, color=color, label=f'Bus {b}')
        all_export[f'bus{b}_PLL_Hz'] = f
    axes[0].set(ylabel='PLL frequency (Hz)', title='All recorded PLL buses: near, intermediate and furthest')
    axes[0].ticklabel_format(axis='y', style='plain', useOffset=False)
    axes[1].set(ylabel='Change from pre-event mean (mHz)', title='Baseline: 0.8 <= t < 1.0 s')
    for ax in axes:
        ax.set(xlabel='Time (s)', xlim=(0,5))
        ax.axvline(1, color='#BA3945', ls='--', lw=1)
        ax.legend(loc='best', fontsize=8)
    fig.suptitle('20 Hz case | All PLL-recorded buses (IBR connection not required)', fontweight='bold')
    for suffix in ['png', 'pdf']:
        fig.savefig(out/f'02_all_pll_bus_distance_extremes.{suffix}', dpi=200, bbox_inches='tight')
    FIGURES['representative_ibr_pll/02_all_pll_bus_distance_extremes']=fig
    plt.close(fig)
    all_export.to_csv(out/'all_pll_representative_frequency.csv', index=False)
    pd.DataFrame(all_selected).to_csv(out/'all_pll_selected_buses.csv', index=False)
    input_records = [
        dict(path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest())
        for path in [raw_path, xlsx_path]
    ]
    feather_path = STREAM_FILES[3]
    input_records.append(
        dict(path=str(feather_path), sha256=hashlib.sha256(feather_path.read_bytes()).hexdigest())
    )
    report = dict(source_bus=4131, distance_definition='Unweighted undirected BFS; every in-service line or two-winding transformer counts as one branch; parallel circuits do not add hops',
                  model_note='RAW supplies bus/line/transformer/zone topology. XLSX is dynamic parameters, not an additional branch topology; regca/ibrepri sheets identify IBR candidates.',
                  bus_count=len(buses), connected_bus_count=len(distance), online_line_records=sum(e['kind']=='line' for e in edges), online_transformer_records=sum(e['kind']=='transformer' for e in edges),
                  pll_channels=len(table), ibr_device_count=len(ibrs), ibr_bus_candidate_count=len(candidates),
                  middle_selection='Closest distance to midpoint of minimum and maximum IBR distances, then smallest bus number; same bus-number tie-break at extrema',
                  all_pll_bus_extremes=extremes, selected_all_pll_buses=all_selected, selected_ibr_buses=chosen,
                  farthest_ibr_ties=candidates[candidates.distance_branches==hi].to_dict('records'),
                  interpretation='Bus PLL channels, shared by IBRs on the same bus; not independent per-device measurements. BFS distance is topological, not impedance or physical distance.',
                  validation='PASS: finite CSV, monotonic time, paths match distances, every graph edge satisfies BFS distance bounds',
                  inputs=input_records)
    (out/'summary.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps({k:v for k,v in report.items() if k != 'inputs'}, indent=2))

In [ ]:
# Forced-oscillation analysis for all 14 RAW zones and source generator 66.
from pathlib import Path
import argparse
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = None  # set by plot_forced_oscillation(output_dir=...)


def read_zones(path):
    zones = {}
    active = False
    for line in path.read_text().splitlines():
        if line.strip().startswith('0 /'):
            active = 'Begin Zone data' in line
            continue
        if active and line.strip():
            values = next(csv.reader([line], quotechar="'", skipinitialspace=True))
            zones[int(values[0])] = values[1].strip()
    if set(zones) != set(range(1, 15)):
        raise ValueError(f'Expected exactly 14 zones, found {zones}')
    return zones


def save(fig, name):
    fig.savefig(OUT / (name + '.png'), dpi=200, bbox_inches='tight')
    fig.savefig(OUT / (name + '.pdf'), bbox_inches='tight')
    FIGURES[name] = fig
    plt.close(fig)


def plot_forced_oscillation(data_dir=None, output_dir=None, support_dir=None, stream_frames=None):
    global OUT
    if data_dir is not None or support_dir is not None or stream_frames is not None:
        configure(data_dir, support_dir, stream_frames)
    if output_dir is not None:
        OUT = Path(output_dir).resolve()
    OUT.mkdir(parents=True, exist_ok=True)
    buses, _, _ = read_raw(AREA_RAW)
    _, machines, _ = read_raw(MACHINE_RAW)
    dynamics = read_dynamic_generators(DYNAMIC_XLSX)
    zones = read_zones(AREA_RAW)
    power, speed = load_csv(5), load_csv(8)
    if list(power.columns) != list(speed.columns) or not np.array_equal(power.time, speed.time):
        raise ValueError('Power/speed channel or time mismatch')
    config = json.loads(static_inputs_path().read_text(encoding='utf-8'))
    fo = config['forced_oscillation']
    if not fo['enabled']:
        raise ValueError('Forced oscillation is disabled')
    start_time = float(fo['start_time'])
    forcing_hz = float(fo['frequency_hz'])
    forcing_label = f'{forcing_hz:g}'
    time = speed.time.to_numpy()
    pre = (time >= start_time - .2) & (time < start_time)
    target = speed.columns.drop('time')[int(fo['target_gen'])]
    if target != 'gen66_bus4131_idH':
        raise ValueError(f'Unexpected forcing target: {target}')
    if config['gen_trip']['enabled'] and float(config['gen_trip']['time']) <= time[-1]:
        raise ValueError('This analysis expects no in-record generator trip')
    records = []
    for channel in speed.columns.drop('time'):
        match = re.fullmatch(r'gen(\d+)_bus(\d+)_id(.+)', channel)
        if not match:
            raise ValueError(f'Unknown channel {channel}')
        key = (int(match[2]), match[3])
        records.append(dict(column=channel, generator_id=key[1], **buses[key[0]],
                            **machines[key], **dynamics[key],
                            zone_name=zones[buses[key[0]]['zone']],
                            pre_event_p_mw=float(power.loc[pre, channel].mean()),
                            mbase_source=MACHINE_RAW.name,
                            zone_source=AREA_RAW.name))
    meta = pd.DataFrame(records).set_index('column', drop=False)
    meta.index.name = 'channel_index'
    if set(dynamics) != set(zip(meta.bus, meta.generator_id)):
        raise ValueError('GENROU channels do not cover XLSX generators exactly')
    if not ((meta.H_s > 0) & (meta.mbase_mva > 0) & (meta.status == 1)).all():
        raise ValueError('Invalid inertia, MBASE or status')
    meta['weight_HS'] = meta.H_s * meta.mbase_mva
    frequency = speed.drop(columns='time') / (2*np.pi)
    if not np.allclose(frequency.loc[pre].median(), 60, atol=.1):
        raise ValueError('Unexpected rotor-speed units')
    coi = pd.DataFrame({'time_s': time})
    rep_frequency = pd.DataFrame({'time_s': time})
    selected, summaries = [], []
    for zone, name in zones.items():
        group = meta[meta.zone == zone]
        column = f'zone{zone:02d}_{name}_Hz'
        has_coi = not group.empty
        if has_coi:
            weights = np.broadcast_to(group.weight_HS.to_numpy(), (len(time), len(group))).copy()
            denominator = weights.sum(axis=1)
            vals = np.divide((frequency[group.index].to_numpy() * weights).sum(axis=1),
                             denominator, out=np.full(len(time), np.nan), where=denominator > 0)
        else:
            vals = np.full(len(time), np.nan)
        coi[column] = vals
        eligible = group[np.isclose(group.kv, 20)]
        representative = None
        if not eligible.empty:
            representative = eligible.sort_values(['pre_event_p_mw', 'column'], ascending=[False, True]).iloc[0]
            selected.append(representative.to_dict())
            rep_frequency[f'zone{zone:02d}_{representative.column}_Hz'] = frequency[representative.column]
        summaries.append(dict(zone=zone, zone_name=name, synchronous_generators=len(group),
                              online_generators=len(group),
                              eligible_20kv_generators=len(eligible), coi_available=has_coi,
                              representative_available=representative is not None,
                              representative=representative.column if representative is not None else '',
                              bus=int(representative.bus) if representative is not None else None,
                              generator_id=representative.generator_id if representative is not None else '',
                              bus_name=representative.bus_name if representative is not None else '',
                              kv=float(representative.kv) if representative is not None else None,
                              H_s=float(representative.H_s) if representative is not None else None,
                              end_coi_hz=float(vals[-1]) if np.isfinite(vals[-1]) else None,
                              limitation='' if has_coi and representative is not None else
                              ('No recorded synchronous generators' if not has_coi else 'No recorded 20 kV synchronous generator')))
    meta.to_csv(OUT/'generator_zone_metadata.csv', index=False)
    coi.to_csv(OUT/'zone_coi_frequency.csv', index=False)
    rep_frequency.to_csv(OUT/'representative_rotor_frequency.csv', index=False)
    pd.DataFrame(selected).to_csv(OUT/'representative_generators.csv', index=False)
    pd.DataFrame(summaries).to_csv(OUT/'zone_summary.csv', index=False)
    plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'axes.spines.top':False,
                         'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':.22,
                         'lines.linewidth':1.7, 'pdf.fonttype':42, 'axes.formatter.useoffset':False})
    colors = list(plt.get_cmap('tab20').colors)
    def format_axis(ax, event_linestyle="--"):
        ax.axvline(start_time, color='#B93B49', lw=1, ls=event_linestyle)
        ax.set(xlim=(0,time[-1]), xlabel='Time (s)', ylabel='Frequency (Hz)')
    fig, ax = plt.subplots(figsize=(13,7), layout='constrained')
    for row in summaries:
        z = row['zone']
        ax.plot(time, coi.iloc[:,z], color=colors[z-1], ls='-',
                label=f'{z:02d} {zones[z]}')
    format_axis(ax, event_linestyle='-')
    ax.set_title('COI frequencies of all 14 zones | H x MBASE weighted', loc='left', fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01,1), fontsize=9)
    save(fig,'01_all_14_zone_coi_frequency')
    fig_coi, axes_coi = plt.subplots(7,2,figsize=(15,20),sharex=True,sharey=True,layout='constrained')
    fig_rep, axes_rep = plt.subplots(7,2,figsize=(15,20),sharex=True,sharey=True,layout='constrained')
    lookup = {r['zone']:r for r in selected}
    individual = OUT/'individual_zones'
    individual.mkdir(exist_ok=True)
    for row, ax_coi, ax_rep in zip(summaries, axes_coi.flat, axes_rep.flat):
        z = row['zone']
        values = coi.iloc[:,z]
        ax_coi.plot(time, values, color=colors[z-1])
        ax_coi.set_title(f'Zone {z:02d}: {zones[z]} ({row["synchronous_generators"]} synchronous generators)',loc='left',fontsize=10)
        format_axis(ax_coi)
        fig, ax = plt.subplots(figsize=(10,4.8),layout='constrained')
        for panel in [ax_rep, ax]:
            panel.plot(time, values, color='#656D78', ls='--',lw=1.4,label='Zone COI')
            if z in lookup:
                rep=lookup[z]
                panel.plot(time, frequency[rep['column']],color=colors[z-1],
                            label=f'Rotor: bus {rep["bus"]}, ID {rep["generator_id"]} (20 kV)')
                panel.set_title(f'Zone {z:02d}: {zones[z]} | {rep["bus_name"]}',loc='left',fontsize=10)
            else:
                panel.set_title(f'Zone {z:02d}: {zones[z]}',loc='left')
                panel.text(.1,.5,row['limitation'],transform=panel.transAxes)
            format_axis(panel)
            panel.legend(loc='lower left',fontsize=8)
        save(fig, f'individual_zones/zone{z:02d}_coi_and_representative')
    fig_coi.suptitle(
        f'14-zone COI frequencies | H x MBASE weighted | Forced oscillation: '
        f'{forcing_label} Hz, start {start_time:g} s', fontweight='bold'
    )
    fig_rep.suptitle(
        f'20 kV representative rotor frequencies and zone COI | Forced oscillation: '
        f'{forcing_label} Hz, start {start_time:g} s', fontweight='bold'
    )
    save(fig_coi,'02_zone_coi_panels')
    save(fig_rep,'03_zone_representative_rotor_frequency')
    report=dict(grouping='RAW BUS field 6 (ZONE); all 14 Zone records',
                zone_source=AREA_RAW.name, inertia_source=DYNAMIC_XLSX.name,
                mbase_source=MACHINE_RAW.name, coi_method='H_s * MBASE weighted',
                forcing_target=target, forcing_start_time_s=start_time, forcing_parameters=fo,
                selection='Highest mean pre-event P among recorded 20 kV GENROU in each zone',
                all_105_dynamic_generators_covered=True, zones=summaries,
                unavailable_zones=[r for r in summaries if r['limitation']])
    (OUT/'summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(pd.DataFrame(summaries).to_string(index=False))
    print('Unavailable zones:', report['unavailable_zones'])
    create_diagnostics(time, power, speed, frequency, coi, meta, target, fo, config, OUT)
    if np.isclose(forcing_hz, 20.0):
        plot_representative_ibr_pll(OUT / 'representative_ibr_pll')

In [ ]:
# Independent numerical and output checks for the saved forced-oscillation analysis.
from pathlib import Path
import argparse
import json
import numpy as np
import pandas as pd


def validate_results(data_dir=None, analysis_dir=None, support_dir=None):
    if data_dir is not None or support_dir is not None:
        configure(data_dir, support_dir)
    folder=Path(analysis_dir).resolve()
    config=json.loads(static_inputs_path().read_text(encoding='utf-8'))
    forcing=config['forced_oscillation']
    start=float(forcing['start_time'])
    hz=float(forcing['frequency_hz'])
    hz_label=f'{hz:g}'
    meta=pd.read_csv(folder/'generator_zone_metadata.csv')
    summary=pd.read_csv(folder/'zone_summary.csv')
    coi=pd.read_csv(folder/'zone_coi_frequency.csv')
    speed=load_csv(8)
    dt_rows=float(np.median(np.diff(speed.time)))
    expected_rows=int(round(float(config['simulation']['t_end'])/dt_rows))
    assert len(speed)==expected_rows, f'{len(speed)} samples, expected {expected_rows}'
    assert len(meta)==105 and len(summary)==14 and coi.shape==(expected_rows,15)
    assert np.isfinite(coi.to_numpy()).all()
    assert summary.coi_available.all() and summary.representative_available.all()
    assert summary.online_generators.sum()==105
    buses,_,_=read_raw(AREA_RAW)
    for row in meta.itertuples():
        assert row.zone==buses[row.bus]['zone']
    for row in summary.itertuples():
        group=meta[meta.zone==row.zone]
        eligible=group[group.kv==20]
        assert row.representative==eligible.sort_values(['pre_event_p_mw','column'],ascending=[False,True]).iloc[0].column
        for target in [.5,1.,1.001,2.,5.]:
            j=int(np.argmin(abs(speed.time-target)))
            expected=np.average(speed.loc[j,group.column].to_numpy(dtype=float)/(2*np.pi),
                                weights=group.H_s*group.mbase_mva)
            assert abs(expected-coi.iloc[j,row.zone])<1e-12
    cycle=pd.read_csv(folder/'source_cycle_means.csv')
    p=load_csv(5);channel='gen66_bus4131_idH'
    dt=float(np.median(np.diff(p.time)))
    expected_samples=int(round(1/(hz*dt)))
    expected_cycles=int(np.floor((p.time.iloc[-1]+dt-start)*hz+1e-8))
    assert len(cycle)==expected_cycles and (cycle.samples==expected_samples).all()
    for row in cycle.itertuples():
        mask=(p.time>=row.start_s-1e-9)&(p.time<row.end_s-1e-9)
        assert mask.sum()==expected_samples
        assert abs(p.loc[mask,channel].mean()-row.target_P_mean_MW)<1e-9
    ss=json.loads((folder/'source_summary.json').read_text())
    pre=(p.time>=start-.2)&(p.time<start)
    late=(p.time>=4)&(p.time<5)
    assert abs(p.loc[late,channel].mean()-p.loc[pre,channel].mean()-ss['observed_late_mean_shift_MW'])<1e-9
    t=np.arange(3000)*.001
    known=7+2*t+3*np.sin(2*np.pi*hz*t+.7)
    fit=harmonic_fit(t,known,hz)
    assert abs(fit['amplitude_peak']-3)<1e-10 and abs(fit['slope_per_s']-2)<1e-10
    freq,amp=amplitude_spectrum(t,3*np.sin(2*np.pi*hz*t))
    j=int(np.argmin(abs(freq-hz)))
    assert abs(amp[j]-3)<1e-4 and abs(freq[j]-hz)<1e-9
    assert len(list((folder/'individual_zones').glob('*.png')))==14
    assert len(list((folder/'individual_zones').glob('*.pdf')))==14
    for name in ['04_bus4131_source_response','05_gen66_cycle_mean_and_oscillation',
                 '06_gen66_response_spectra',f'07_regional_{hz_label}Hz_response']:
        assert (folder/(name+'.png')).stat().st_size>10000
        assert (folder/(name+'.pdf')).stat().st_size>10000
    extra_ibr_pll=False
    if np.isclose(hz,20.0):
        pll=folder/'representative_ibr_pll'
        for name in ['01_near_middle_far_ibr_pll','02_all_pll_bus_distance_extremes']:
            assert (pll/(name+'.png')).stat().st_size>10000
            assert (pll/(name+'.pdf')).stat().st_size>10000
        for name in ['all_pll_bus_distances.csv','in_service_topology_edges.csv',
                     'ibr_bus_candidates.csv','selected_representatives.csv',
                     'selected_pll_frequency.csv','all_pll_representative_frequency.csv',
                     'all_pll_selected_buses.csv','summary.json']:
            assert (pll/name).is_file()
        extra_ibr_pll=True
    result=dict(status='PASS',raw_zone_assignment_checks=105,independent_coi_checks=70,
                forcing_frequency_hz=hz,complete_cycle_mean_checks=expected_cycles,
                cycle_samples=expected_samples,extra_ibr_pll_outputs=extra_ibr_pll,
                all_14_zones_have_20kv_representatives=True,all_105_generators_remain_online=True,
                mean_shift_verified=True,harmonic_fit_and_fft_amplitude_calibration=True)
    (folder/'validation.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    print(json.dumps(result,indent=2))

## Run the analysis

In [ ]:
configure(data_dir=PARAEMT_BUILD_DIR, support_dir=MODEL_DIR, stream_frames=stream_frames)
plot_forced_oscillation(output_dir=OUTPUT_DIR)

print('Plot generation complete.')

In [ ]:
validate_results(analysis_dir=OUTPUT_DIR)

validation = json.loads((OUTPUT_DIR / 'validation.json').read_text(encoding='utf-8'))
display(Markdown('## Validation result'))
display(pd.Series(validation, name='value').to_frame())

In [ ]:
forcing_label = f"{float(paraemt['parameters']['forced_oscillation']['frequency_hz']):g}"
display(Markdown(f'## Main {forcing_label} Hz plots'))
main_plot_names = [
    '01_all_14_zone_coi_frequency',
    '02_zone_coi_panels',
    '03_zone_representative_rotor_frequency',
    '04_bus4131_source_response',
    '05_gen66_cycle_mean_and_oscillation',
    '06_gen66_response_spectra',
    f'07_regional_{forcing_label}Hz_response',
    '08_gen66_active_power_vs_time',
]
for name in main_plot_names:
    display(Markdown(f'### `{name}.png`'))
    display(FIGURES[name])

In [ ]:
pll_dir = OUTPUT_DIR / 'representative_ibr_pll'
if pll_dir.is_dir():
    display(Markdown(f'## Additional {forcing_label} Hz IBR/PLL plots'))
    for name in ['01_near_middle_far_ibr_pll', '02_all_pll_bus_distance_extremes']:
        display(Markdown(f'### `{name}.png`'))
        display(FIGURES[f'representative_ibr_pll/{name}'])

display(Markdown('## Key derived tables'))
regional = pd.read_csv(OUTPUT_DIR / f'regional_{forcing_label}Hz_response.csv')
display(Markdown(f'### Regional {forcing_label} Hz response'))
display(regional)
if pll_dir.is_dir():
    representatives = pd.read_csv(pll_dir / 'selected_representatives.csv')
    display(Markdown('### Near/middle/far IBR representatives'))
    display(representatives)

In [ ]:
display(Markdown('## Individual zone plots'))
for zone in range(1, 15):
    name = f'individual_zones/zone{zone:02d}_coi_and_representative'
    display(Markdown(f'### `{name}.png`'))
    display(FIGURES[name])

In [ ]:
output_inventory = pd.DataFrame([
    {
        'relative_path': str(path.relative_to(OUTPUT_DIR)),
        'type': path.suffix.lower(),
        'size_kB': path.stat().st_size / 1000,
    }
    for path in sorted(OUTPUT_DIR.rglob('*'))
    if path.is_file()
])
display(Markdown(f'## Output inventory ({len(output_inventory)} files)'))
display(output_inventory)